# Motivation

- For many NLP applications involving Transformer models, you can simply take a pretrained model from the Hugging Face Hub and fine-tune it directly on your data for the task at hand.
    - Provided that the corpus used for pretraining is not too different from the corpus used for fine-tuning, transfer learning will usually produce good results.

- However, there are a few cases where you’ll want to first fine-tune the language models on your data, before training a task-specific head.
- For example, if your dataset contains legal contracts or scientific articles, a vanilla Transformer model like BERT will typically treat the domain-specific words in your corpus as rare tokens, and the resulting performance may be less than satisfactory.
- By fine-tuning the language model on in-domain data you can boost the performance of many downstream tasks, which means you usually only have to do this step once!
- This process of fine-tuning a pretrained language model on in-domain data is usually called **domain adaptation**.
- It was popularized in 2018 by [ULMFiT](https://arxiv.org/abs/1801.06146), which was one of the first neural architectures (based on LSTMs) to make transfer learning really work for NLP.

# Load deps

In [ ]:
import os, torch, collections
import numpy as np
from datasets import load_dataset
from kaggle_secrets import UserSecretsClient
from transformers import AutoModelForMaskedLM
from transformers import AutoTokenizer
from transformers import DataCollatorForLanguageModeling
from transformers import default_data_collator
from transformers import TrainingArguments, Trainer

# HF Login

In [ ]:
secret_label = "HF_TOKEN"
os.environ[secret_label] = UserSecretsClient().get_secret(secret_label)
!hf auth whoami

# Config

## Model selection
- you can find a list of candidates by applying the “Fill-Mask” filter on the [Hugging Face Hub](https://huggingface.co/models?pipeline_tag=fill-mask&sort=downloads)
- Although the BERT and RoBERTa family of models are the most downloaded, we’ll use a model called [DistilBERT](https://huggingface.co/distilbert-base-uncased) that can be trained much faster with little to no loss in downstream performance.
- This model was trained using a special technique called [_knowledge distillation_](https://en.wikipedia.org/wiki/Knowledge_distillation)
    - where a large “teacher model” like BERT is used to guide the training of a “student model” that has far fewer parameters.
    - An explanation of the details of knowledge distillation can be found in [_Natural Language Processing with Transformers_](https://www.oreilly.com/library/view/natural-language-processing/9781098136789/) (colloquially known as the Transformers textbook).

In [ ]:
hf_user_name = "amin-oj"
model_checkpoint = "distilbert-base-uncased"

dataset_dict = {
    "path": "imdb",
}
push_to_hub=True
train_bs = 64
eval_bs = 64
num_train_epochs = 3
ckpt_name = f"{model_checkpoint.split("/")[-1]}-finetuned-mlm-{dataset_dict['path']}"
print(ckpt_name)

# Load and test pretrained checkpoint

In [ ]:
model = AutoModelForMaskedLM.from_pretrained(model_checkpoint)
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

In [ ]:
text = "This is a great [MASK]."
inputs = tokenizer(text, return_tensors="pt")
token_logits = model(**inputs).logits
# Find the location of [MASK] and extract its logits
mask_token_index = torch.where(inputs["input_ids"] == tokenizer.mask_token_id)[1]
mask_token_logits = token_logits[0, mask_token_index, :]
# Pick the [MASK] candidates with the highest logits
top_5_tokens = torch.topk(mask_token_logits, 5, dim=1).indices[0].tolist()
for token in top_5_tokens:
    print(f"'>>> {text.replace(tokenizer.mask_token, tokenizer.decode([token]))}'")

- As humans, we can imagine many possibilities for the `[MASK]` token, such as “day”, “ride”, or “painting”.
- For pretrained models, the predictions depend on the corpus the model was trained on, since it learns to pick up the statistical patterns present in the data.
- Like BERT, DistilBERT was pretrained on the [English Wikipedia](https://huggingface.co/datasets/wikipedia) and [BookCorpus](https://huggingface.co/datasets/bookcorpus) datasets, so we expect the predictions for `[MASK]` to reflect these domains.
- We can see from the outputs that the model’s predictions refer to everyday terms, which is perhaps not surprising given the foundation of English Wikipedia.
- Let’s see how we can change this domain to something a bit more niche — highly polarized movie reviews!

# Load & inspect data

- We’ll use the famous [Large Movie Review Dataset](https://huggingface.co/datasets/imdb) (or IMDb for short)
- which is a corpus of movie reviews that is often used to benchmark sentiment analysis models.
- By fine-tuning DistilBERT on this corpus, we expect the language model will adapt its vocabulary
    - from the factual data of Wikipedia that it was pretrained on to the more subjective elements of movie reviews.

In [ ]:
imdb_dataset = load_dataset(**dataset_dict)

In [ ]:
print(imdb_dataset)
print("=================")
sample = imdb_dataset["train"].shuffle(seed=42).select(range(4))
# sample = imdb_dataset["unsupervised"].shuffle(seed=42).select(range(4))
for row in sample:
    print(f"\n>>> Review: {row['text'][:100]}...")
    print(f">>> Label: {row['label']}")
print("=================")
print(imdb_dataset["train"].features["label"].names)
print("=================")

# Prep data

In [ ]:
# keep only the `train` split to speed up the rest of the code
del imdb_dataset["test"]
del imdb_dataset["unsupervised"]

- For both auto-regressive and masked language modeling, a common preprocessing step is
    - to concatenate all the examples
    - and then split the whole corpus into chunks of equal size.
- This is quite different from our usual approach, where we simply tokenize individual examples.
- Why **concatenate** everything together?
    - The reason is that individual examples might get truncated if they’re too long
    - and that would result in losing information that might be useful for the language modeling task!

- We’ll first tokenize our corpus as usual, but **without** setting the `truncation=True` option in our tokenizer.
- We’ll also grab the word IDs if they are available (i.e. if we’re using a fast tokenizer)
    - as we will need them later on to do whole word masking.
- We’ll remove the `text` and `label` columns since we don’t need them any longer

In [ ]:
def tokenize_function(examples, tokenizer):
    result = tokenizer(examples["text"])
    if tokenizer.is_fast:
        result["word_ids"] = [result.word_ids(i) for i in range(len(result["input_ids"]))]
    return result

# tkres = tokenizer(imdb_dataset["train"][:5]["text"])
# [tkres.word_ids(i) for i in range(len(tkres["input_ids"]))]

tokenized_datasets = imdb_dataset.map(
    tokenize_function,
    batched=True,
    fn_kwargs = {"tokenizer": tokenizer},
    remove_columns=["text", "label"]
)

In [ ]:
print(tokenized_datasets)

- Now that we’ve tokenized our movie reviews, the next step is to group them all together and split the result into chunks.
- But how big should these chunks be?
    - This will ultimately be determined by the amount of GPU memory that you have available,
    - but a good starting point is to see what the model’s maximum context size is.
        - This can be inferred by inspecting the `model_max_length` attribute of the tokenizer
        - This value is derived from the _tokenizer\_config.json_ file associated with a checkpoint
    - in order to run our experiments on GPUs like those found on Google Colab, we’ll pick something a bit smaller that can fit in memory.
        - Note that using a small chunk size can be detrimental in real-world scenarios
        - so you should use a size that corresponds to the use case you will apply your model to.

In [ ]:
print(f"model max_seq_len: {tokenizer.model_max_length}")
chunk_size = tokenizer.model_max_length // 4
print(f"selected chunk size: {chunk_size}")

## Concat & chunkify

In [ ]:
tokenized_samples = tokenized_datasets["train"][:3]
for idx, sample in enumerate(tokenized_samples["input_ids"]):
    print(f"'>>> Review {idx} length: {len(sample)}'")
print("=================")

concatenated_examples = {
    k: sum(tokenized_samples[k], []) for k in tokenized_samples.keys()
}
total_length = len(concatenated_examples["input_ids"])
print(f"'>>> Concatenated reviews length: {total_length}'")
print("=================")


chunks = {
    k: [t[i : i + chunk_size] for i in range(0, total_length, chunk_size)]
    for k, t in concatenated_examples.items()
}

for chunk in chunks["input_ids"]:
    print(f"'>>> Chunk length: {len(chunk)}'")
print("=================")

- As you can see in this example, the last chunk will generally be smaller than the maximum chunk size. There are two main strategies for dealing with this:
    - Drop the last chunk if it’s smaller than `chunk_size`.
    - Pad the last chunk until its length equals `chunk_size`.
    - We’ll take the first approach here
- Let’s wrap all of the above logic in a single function that we can apply to our tokenized datasets.
---
Note: We’re missing a key step: inserting `[MASK]` tokens at random positions in the inputs
- We'll do this on the fly during fine-tuning using a special data collator.

In [ ]:
def group_texts(examples):
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    total_length = (total_length // chunk_size) * chunk_size # drop last
    # TODO: Shouldn't we add an overlap between slices?
    # TODO: How does it impact the final result??
    result = {
        k: [t[i : i + chunk_size] for i in range(0, total_length, chunk_size)]
        for k, t in concatenated_examples.items()
    }
    result["labels"] = result["input_ids"].copy() # MLM labels
    return result

lm_datasets = tokenized_datasets.map(group_texts, batched=True)

In [ ]:
print(lm_datasets)

# Fine-tuning with the Trainer API

- Fine-tuning a masked language model is almost identical to fine-tuning a sequence classification model
- The only difference is that we need a special data collator that can randomly mask some of the tokens in each batch of texts.
- Fortunately, 🤗 Transformers comes prepared with a dedicated `DataCollatorForLanguageModeling` for just this task.
    - We just have to pass it the tokenizer and an `mlm_probability` argument that specifies what fraction of the tokens to mask.
    - We’ll pick 15%, which is the amount used for BERT and a common choice in the literature.
- One side effect of random masking is that our evaluation metrics will not be deterministic when using the Trainer, since we use the same data collator for the training and test sets.
    - With 🤗 Accelerate, we can use the flexibility of a custom evaluation loop to freeze the randomness.

In [ ]:
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm_probability=0.15)

samples = [lm_datasets["train"][i] for i in range(2)]
for sample in samples: _ = sample.pop("word_ids") # this DL doesn't expect this key
collated_samples = data_collator(samples)
for chunk in collated_samples["input_ids"]:
    print(f"\n'>>> {tokenizer.decode(chunk)}'")
print("================")
for label in collated_samples["labels"]: print(label)
print("================")

## Whole word masking

- When training models for masked language modeling, one technique that can be used is to mask whole words together, not just individual tokens.
- This approach is called _whole word masking_.
- If we want to use whole word masking, we will need to build a data collator ourselves.
    - A data collator is just a function that takes a list of samples and converts them into a batch.
- Note that the labels are all -100 except for the ones corresponding to mask words.

In [ ]:
def whole_word_masking_data_collator(features, mak_prob = 0.2):
    for feature in features:
        word_ids = feature.pop("word_ids")

        # Create a map between words and corresponding token indices
        mapping = collections.defaultdict(list)
        current_word_index = -1
        current_word = None
        for idx, word_id in enumerate(word_ids):
            if word_id is not None:
                if word_id != current_word:
                    current_word = word_id
                    current_word_index += 1
                mapping[current_word_index].append(idx)

        # Randomly mask words
        mask = np.random.binomial(1, mak_prob, (len(mapping),))
        input_ids = feature["input_ids"]
        labels = feature["labels"]
        new_labels = [-100] * len(labels)
        for word_id in np.where(mask)[0]:
            word_id = word_id.item()
            for idx in mapping[word_id]:
                new_labels[idx] = labels[idx]
                input_ids[idx] = tokenizer.mask_token_id
        feature["labels"] = new_labels

    return default_data_collator(features)

In [ ]:
samples = [lm_datasets["train"][i] for i in range(2)]

# tks = tokenizer.convert_ids_to_tokens(samples[0]['input_ids'])
# w_ids = samples[0]['word_ids']
# for t,w in zip(tks, w_ids): print(f"word {w} | token: {t}")

ww_collated_samples = whole_word_masking_data_collator(samples)
for chunk in ww_collated_samples["input_ids"]:
    print(f"\n'>>> {tokenizer.decode(chunk)}'")
print("================")
for label in ww_collated_samples["labels"]: print(label)
print("================")

In [ ]:
# downsize the dataset to speed up training
train_size = 10_000
test_size = int(0.1 * train_size)
downsampled_dataset = lm_datasets["train"].train_test_split(
    train_size=train_size, test_size=test_size, seed=42
)
print(downsampled_dataset)

- By default, the `Trainer` will remove any columns that are not part of the model’s `forward()` method.
- This means that if you’re using the whole word masking collator, you’ll also need to set `remove_unused_columns=False` to ensure we don’t lose the `word_ids` column during training.

In [ ]:
training_args = TrainingArguments(
    output_dir=ckpt_name,
    per_device_train_batch_size=train_bs,
    per_device_eval_batch_size=eval_bs,
    push_to_hub=push_to_hub,
    num_train_epochs=num_train_epochs,
    logging_steps=len(downsampled_dataset["train"]) // train_bs,
    eval_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    bf16=True, # better than fp16 if the device supports
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=downsampled_dataset["train"],
    eval_dataset=downsampled_dataset["test"],
    data_collator=data_collator,
    # TODO: test whole word collator and compare the results
    processing_class=tokenizer,
)

## Perplexity for language models

- Assuming our test set consists mostly of sentences that are grammatically correct, then one way to measure the quality of our language model is to calculate the probabilities it assigns to the next word in all the sentences of the test set.
- High probabilities indicates that the model is not “surprised” or “perplexed” by the unseen examples, and suggests it has learned the basic patterns of grammar in the language.
- There are various mathematical definitions of perplexity, but the one we’ll use defines it as the exponential of the cross-entropy loss.

In [ ]:
import math
eval_results = trainer.evaluate()
print(f">>> Perplexity: {math.exp(eval_results['eval_loss']):.2f}")

In [ ]:
trainer.train()

In [ ]:
eval_results = trainer.evaluate()
print(f">>> Perplexity: {math.exp(eval_results['eval_loss']):.2f}")

In [ ]:
trainer.push_to_hub()

# Using our fine-tuned model

In [ ]:
from transformers import pipeline

mask_filler = pipeline(
    "fill-mask", model=f"{hf_user_name}/{ckpt_name}"
)

text = "This is a great [MASK]."

preds = mask_filler(text)

for pred in preds:
    print(f">>> {pred['sequence']}")